In [0]:
import pyspark.sql.functions as F

In [0]:
dbutils.widgets.text('Movie Name','The Shawshank Redemption')

In [0]:
movie_name=dbutils.widgets.get('Movie Name')

In [0]:
def movie_recommender_engine(movie_name):
    article=['a','an','the']
    word=movie_name.lower().split()
    if word[0] in article:
        target=word[0]
        word.remove(target)
        result=' '.join(word)+', '+target
    else:
        result = movie_name.lower()
    genre_list=spark.sql(f"SELECT DISTINCT genre FROM movie_recommender.silver.movies WHERE LOWER(title) LIKE LOWER('%{result}%')").collect()
    genre_list=[row.genre for row in genre_list]
    if not genre_list:
        return "Movie not found in database. Please try another title."
    condition = ' OR '.join([f"array_contains(genres, '{genre}')" for genre in genre_list])
    top_10_recommendations=spark.sql(f"SELECT * FROM movie_recommender.gold.popular_movies WHERE ({condition}) AND LOWER(title) NOT LIKE LOWER('%{movie_name}%') ORDER BY weighted_rating DESC LIMIT 10")
    return top_10_recommendations